# DeepFake Image Detection Using Transfer Learning (EfficientNetB0)

Fixed, cleaned, and reordered end-to-end notebook: dataset download → corrected train/val/test split → data pipelines → model → training (frozen, then fine-tuned) → evaluation → Streamlit app.

**What was fixed vs. the previous version:**
- Cell 2 (dataset split) now checks only the *immediate parent folder name* of each image instead of the full file path, which previously caused every image to be misclassified as "fake" (0 real images).
- Old dataset directory is wiped before rebuilding the split, so no stale/broken files linger.
- A sanity-check cell (per-class image counts) now runs immediately after the split, before any training happens, so a broken split is caught early instead of discovered after a full training run.
- Removed a duplicate/out-of-order debug cell that referenced `y_true`/`y_pred` before they were defined (this was causing a `NameError`).
- Confusion matrix, classification report, and ROC curve are now in one clean cell that defines its own variables.

## Step 1: Download the dataset

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("shivamardeshna/real-and-fake-images-dataset-for-image-forensics")

print("Path to dataset files:", path)


## Step 2: Explore the downloaded dataset structure

In [ ]:
import os

for root, dirs, files_ in os.walk(path):
    level = root.replace(path, '').count(os.sep)
    indent = '  ' * level
    print(f"{indent}{os.path.basename(root)}/  ({len(files_)} files)")
    if level > 2:
        continue


## Step 3: Build the train/validation/test split (FIXED)

**The fix:** the previous version checked `'fake' in full_path.lower()`, which matched on the dataset's parent directory names and silently dumped every image into the "fake" bucket (0 real images). This version checks only the *immediate parent folder name* of each file, and wipes any old broken split first.

In [ ]:
import os, shutil, random
from pathlib import Path

random.seed(42)

SRC = path  # from kagglehub
DEST = "/content/dataset"
SAMPLES_PER_CLASS = 2500   # per class, before the train/val/test split
SPLIT = {"train": 0.7, "validation": 0.15, "test": 0.15}

# Wipe any old (broken) split before rebuilding
shutil.rmtree(DEST, ignore_errors=True)

real_imgs, fake_imgs = [], []
for root, _, files_ in os.walk(SRC):
    parent_folder = os.path.basename(root).lower()  # only the immediate folder name, not the full path
    for f in files_:
        if f.lower().endswith(('.jpg', '.jpeg', '.png')):
            full_path = os.path.join(root, f)
            if 'fake' in parent_folder:
                fake_imgs.append(full_path)
            elif 'real' in parent_folder:
                real_imgs.append(full_path)

print(f"Found {len(real_imgs)} real images, {len(fake_imgs)} fake images")

random.shuffle(real_imgs)
random.shuffle(fake_imgs)
real_imgs = real_imgs[:SAMPLES_PER_CLASS]
fake_imgs = fake_imgs[:SAMPLES_PER_CLASS]

def split_and_copy(img_list, label):
    n = len(img_list)
    n_train = int(n * SPLIT["train"])
    n_val = int(n * SPLIT["validation"])
    splits = {
        "train": img_list[:n_train],
        "validation": img_list[n_train:n_train+n_val],
        "test": img_list[n_train+n_val:]
    }
    for split_name, imgs in splits.items():
        dest_dir = Path(DEST) / split_name / label
        dest_dir.mkdir(parents=True, exist_ok=True)
        for i, img_path in enumerate(imgs):
            shutil.copy(img_path, dest_dir / f"{i:05d}.jpg")
    print(f"{label}: train={len(splits['train'])}, val={len(splits['validation'])}, test={len(splits['test'])}")

split_and_copy(real_imgs, "real")
split_and_copy(fake_imgs, "fake")

print("\nDone! Dataset ready at /content/dataset")


### Sanity check — run this BEFORE building data loaders

Both classes should show non-zero, roughly balanced counts. If `real` shows 0 anywhere here, stop and re-check Step 3 before continuing — don't proceed to training on a broken split.

In [ ]:
import os
for split in ["train", "validation", "test"]:
    for label in ["real", "fake"]:
        p = f"/content/dataset/{split}/{label}"
        n = len(os.listdir(p)) if os.path.exists(p) else "MISSING"
        print(f"{split}/{label} -> {n}")


## Step 4: Build the `tf.data` pipelines

In [ ]:
import tensorflow as tf

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = "/content/dataset"

train_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/train",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=True,
    seed=42
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/validation",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=False
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    f"{DATA_DIR}/test",
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='binary',
    shuffle=False
)

class_names = train_ds.class_names
print("Class names (0, 1):", class_names)


## Step 5: Preprocessing + data augmentation (training only)

In [ ]:
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomBrightness(0.1),
])

preprocess_input = tf.keras.applications.efficientnet.preprocess_input

def prepare(ds, augment=False):
    ds = ds.map(lambda x, y: (preprocess_input(x), y), num_parallel_calls=tf.data.AUTOTUNE)
    if augment:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.prefetch(tf.data.AUTOTUNE)

train_ds = prepare(train_ds, augment=True)
val_ds = prepare(val_ds)
test_ds = prepare(test_ds)

print("Data pipelines ready.")


## Step 6: Build the model (EfficientNetB0, transfer learning)

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models

base_model = EfficientNetB0(
    include_top=False,
    weights='imagenet',
    input_shape=(224, 224, 3)
)
base_model.trainable = False  # freeze backbone initially

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(1, activation='sigmoid')(x)

model = models.Model(inputs, outputs)
model.summary()


## Step 7: Compile

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)


## Step 8: Callbacks

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import os

os.makedirs("/content/model", exist_ok=True)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ModelCheckpoint("/content/model/best_model.keras", monitor='val_accuracy', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7)
]


## Step 9: Train — Phase 1 (frozen backbone)

In [ ]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)


## Step 10: Fine-tune — Phase 2 (unfreeze top 20 layers)

In [ ]:
base_model.trainable = True

# Freeze all layers except the last 20
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),  # much lower LR for fine-tuning
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Precision(name='precision'), tf.keras.metrics.Recall(name='recall')]
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks
)


## Step 11: Evaluate on the test set

In [ ]:
test_loss, test_acc, test_prec, test_recall = model.evaluate(test_ds)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Precision: {test_prec:.4f}")
print(f"Test Recall: {test_recall:.4f}")


## Step 12: Confusion matrix, classification report, ROC curve

This cell defines `y_true` / `y_pred` itself, in order, so there's no more `NameError` from a stray earlier cell.

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_true = np.concatenate([y.numpy() for x, y in test_ds])
y_pred_proba = model.predict(test_ds).flatten()
y_pred = (y_pred_proba > 0.5).astype(int)

print("Unique y_true:", np.unique(y_true))
print("Unique y_pred:", np.unique(y_pred))
print("Class names:", class_names)

# Confusion matrix
cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(cmap='Blues')
plt.title("Confusion Matrix")
plt.savefig("/content/confusion_matrix.png")
plt.show()

# Classification report
print(classification_report(y_true, y_pred, target_names=class_names))

# ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc='lower right')
plt.savefig("/content/roc_curve.png")
plt.show()


## Step 13: Training / validation curves (for your report)

In [ ]:
def plot_history(history, history_fine=None):
    acc = history.history['accuracy']
    val_acc = history.history['val_accuracy']
    loss = history.history['loss']
    val_loss = history.history['val_loss']

    if history_fine:
        acc = acc + history_fine.history['accuracy']
        val_acc = val_acc + history_fine.history['val_accuracy']
        loss = loss + history_fine.history['loss']
        val_loss = val_loss + history_fine.history['val_loss']

    plt.figure(figsize=(12, 4))
    plt.subplot(1, 2, 1)
    plt.plot(acc, label='Train Accuracy')
    plt.plot(val_acc, label='Val Accuracy')
    plt.legend()
    plt.title('Accuracy')

    plt.subplot(1, 2, 2)
    plt.plot(loss, label='Train Loss')
    plt.plot(val_loss, label='Val Loss')
    plt.legend()
    plt.title('Loss')
    plt.savefig("/content/training_curves.png")
    plt.show()

plot_history(history, history_fine)


## Step 14: Save the final model

In [ ]:
model.save("/content/model/deepfake_model.keras")
print("Saved.")


## Step 15: Streamlit app

**Important:** double-check that `class_names` below matches what Step 4 printed for your run (it should be `['fake', 'real']` given the folder naming, but confirm before trusting the mapping).

In [ ]:
%%writefile app.py
import streamlit as st
import tensorflow as tf
import numpy as np
from PIL import Image

st.set_page_config(page_title="DeepFake Detector", page_icon="🕵️", layout="centered")

@st.cache_resource
def load_model():
    return tf.keras.models.load_model("model/deepfake_model.keras")

model = load_model()
class_names = ['fake', 'real']  # confirm this matches the class_names printed in Step 4

st.title("🕵️ DeepFake Image Detector")
st.write("Upload a facial image to check whether it's **Real** or **AI-generated (Fake)**.")

uploaded_file = st.file_uploader("Choose an image...", type=["jpg", "jpeg", "png"])

if uploaded_file is not None:
    image = Image.open(uploaded_file).convert("RGB")
    st.image(image, caption="Uploaded Image", use_column_width=True)

    if st.button("Detect"):
        img = image.resize((224, 224))
        img_array = np.array(img)
        img_array = tf.keras.applications.efficientnet.preprocess_input(img_array)
        img_array = np.expand_dims(img_array, axis=0)

        prediction = model.predict(img_array)[0][0]
        predicted_class = class_names[1] if prediction > 0.5 else class_names[0]
        confidence = prediction if prediction > 0.5 else 1 - prediction

        st.subheader(f"Prediction: **{predicted_class.upper()}**")
        st.write(f"Confidence: **{confidence*100:.2f}%**")

        if predicted_class == 'fake':
            st.warning("⚠️ This image appears to be AI-generated / manipulated.")
        else:
            st.success("✅ This image appears to be authentic.")


## Step 16: Launch the app

In [ ]:
import os
os.makedirs("model", exist_ok=True)


In [ ]:
!cp /content/model/deepfake_model.keras model/
!pip install -q streamlit
!npm install -q localtunnel


In [ ]:
!streamlit run app.py &>/content/logs.txt &
import time
time.sleep(5)

!npx localtunnel --port 8501
